# Подготовка данных для Investment Advisor
Этот ноутбук загружает сырые данные OHLCV, макроэкономические показатели и новости, вычисляет технические индикаторы, создаёт таргеты и объединяет всё в один датасет.

In [1]:
import sys
import shutil
from datasets import load_dataset
from transformers import pipeline
from tqdm import tqdm
import kagglehub
import torch
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
from io import BytesIO
import logging
import warnings

warnings.filterwarnings('ignore')

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

d:\Storage\Projects\dpo\dpo\project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
BASE_DIR = Path.cwd().parent.parent
RAW_DIR = BASE_DIR / "bigdata" / "raw"
PROCESSED_DIR = BASE_DIR / "bigdata" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_START = "2010-01-01"
DEFAULT_END = "2024-12-31"

RSI_PERIOD = 14
BB_PERIOD = 20
ATR_PERIOD = 14
MACD_FAST = 12
MACD_SLOW = 26
MACD_SIGNAL = 9
STOCH_PERIOD = 14
STOCH_SMOOTH = 3
CCI_PERIOD = 20
MOMENTUM_PERIOD = 20
VOL_PERIOD_SHORT = 20
VOL_PERIOD_LONG = 50
SMA_PERIODS = [20, 50, 200]

_SOAP_URL = "https://www.cbr.ru/DailyInfoWebServ/DailyInfo.asmx"
_HEADERS_XML = {
    "User-Agent": "Mozilla/5.0",
    "Content-Type": "text/xml; charset=utf-8",
}

In [4]:
#Скачивание OHLCV с Kaggle
path = Path(kagglehub.dataset_download("olegshpagin/russia-stocks-prices-ohlcv"))
d1_dir = path / "D1"
src_dir = d1_dir if d1_dir.exists() else path
for csv_file in src_dir.glob("*.csv"):
    shutil.copy2(csv_file, RAW_DIR / csv_file.name)

In [5]:
# 1. USD/RUB
def fetch_usd_rub(start: str = DEFAULT_START, end: str = DEFAULT_END) -> pd.Series:
    """Курс USD/RUB через XML_dynamic.asp ЦБРФ."""
    result = pd.Series(dtype=float)
    current = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)

    while current <= end_ts:
        chunk_end = min(current + pd.Timedelta(days=365), end_ts)
        d1 = current.strftime("%d/%m/%Y")
        d2 = chunk_end.strftime("%d/%m/%Y")
        url = (
            f"http://www.cbr.ru/scripts/XML_dynamic.asp"
            f"?date_req1={d1}&date_req2={d2}&VAL_NM_RQ=R01235"
        )
        try:
            resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
            resp.raise_for_status()
            xml = resp.content.decode("windows-1251")
            root = ET.fromstring(xml)
            for rec in root.findall("Record"):
                date_str = rec.get("Date")
                val = rec.find("Value")
                if date_str and val is not None:
                    dt = pd.to_datetime(date_str, format="%d.%m.%Y")
                    rate = float(val.text.replace(",", "."))
                    result[dt] = rate
        except Exception as exc:
            logger.error("Ошибка загрузки USD/RUB %s-%s: %s", d1, d2, exc)
        current = chunk_end + pd.Timedelta(days=1)

    return result.sort_index()

In [ ]:
# 2. Ключевая ставка — SOAP KeyRateXML
def fetch_key_rate(start: str = DEFAULT_START, end: str = DEFAULT_END) -> pd.Series:
    """Ключевая ставка ЦБРФ через SOAP KeyRateXML. Доступна с 17.09.2013."""
    soap_body = f"""<?xml version="1.0" encoding="utf-8"?>
<soap:Envelope xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
               xmlns:xsd="http://www.w3.org/2001/XMLSchema"
               xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <KeyRateXML xmlns="http://web.cbr.ru/">
      <fromDate>{start}</fromDate>
      <ToDate>{end}</ToDate>
    </KeyRateXML>
  </soap:Body>
</soap:Envelope>"""

    headers = {**_HEADERS_XML, "SOAPAction": "http://web.cbr.ru/KeyRateXML"}
    resp = requests.post(_SOAP_URL, data=soap_body, headers=headers, timeout=60)
    root = ET.fromstring(resp.text)

    results = []
    cur_date = None
    for elem in root.iter():
        tag = elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag
        if tag == "DT":
            cur_date = elem.text[:10]
        elif tag == "Rate" and cur_date:
            results.append((cur_date, float(elem.text)))
            cur_date = None

    df = pd.DataFrame(results, columns=["date", "key_rate"])
    df["date"] = pd.to_datetime(df["date"])
    return df.set_index("date")["key_rate"].sort_index()

In [7]:
# 3. Инфляция — Excel UniDbQuery
def _parse_inflation_date(val):
    """Парсит дату вида 'MM.YYYY' (float или str)."""
    if pd.isna(val):
        return pd.NaT
    parts = str(val).split(".")
    if len(parts) != 2:
        return pd.NaT
    month = int(parts[0])
    year = int(parts[1])
    if year < 100:
        year += 2000
    try:
        return pd.Timestamp(year=year, month=month, day=1)
    except ValueError:
        return pd.NaT

def fetch_inflation(start: str = DEFAULT_START, end: str = DEFAULT_END) -> pd.Series:
    """Инфляция % г/г через Excel-экспорт UniDbQuery (ID 132934)."""
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    url = (
        f"https://www.cbr.ru/Queries/UniDbQuery/DownloadExcel/132934"
        f"?FromDate={s.strftime('%m/%d/%Y')}"
        f"&ToDate={e.strftime('%m/%d/%Y')}"
        f"&posted=False"
    )
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    resp.raise_for_status()

    df = pd.read_excel(BytesIO(resp.content))
    df["date"] = df["Дата"].apply(_parse_inflation_date)
    df = df[(df["date"] >= s) & (df["date"] <= e)]
    return df.set_index("date")["Инфляция, % г/г"].sort_index()

In [8]:
# 4. Денежная масса M2 — Excel
def fetch_m2(start: str = DEFAULT_START, end: str = DEFAULT_END) -> pd.Series:
    """
    Денежная масса M2 (млрд руб., ежемесячно).
    Excel: https://www.cbr.ru/vfs/statistics/credit_statistics/monetary_agg.xlsx
    """
    url = "https://www.cbr.ru/vfs/statistics/credit_statistics/monetary_agg.xlsx"

    try:
        resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        if resp.status_code == 200 and len(resp.content) > 10_000:
            data = resp.content
        else:
            raise ValueError(f"HTTP {resp.status_code} / {len(resp.content)} bytes")
    except Exception as exc:
        logger.warning("Прямая загрузка M2 не удалась (%s), пробуем кэш", exc)
        cached = RAW_DIR / "monetary_agg.xlsx"
        if cached.exists():
            data = cached.read_bytes()
        else:
            raise FileNotFoundError(
                "Нет доступа к M2-данным и нет кэшированного файла. "
                "Скачайте https://www.cbr.ru/vfs/statistics/credit_statistics/monetary_agg.xlsx "
                f"вручную и поместите в {cached}"
            )

    raw = pd.read_excel(BytesIO(data), header=None)
    dates = pd.to_datetime(raw.iloc[0, 1:])
    m2_vals = pd.to_numeric(raw.iloc[13, 1:], errors="coerce")

    df = pd.DataFrame({"date": dates, "m2": m2_vals}).dropna()
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    df = df[(df["date"] >= s) & (df["date"] <= e)]
    return df.set_index("date")["m2"].sort_index()

In [9]:
# Сборка итогового датафрейма
def prepare_macro_data(
    start: str = DEFAULT_START,
    end: str = DEFAULT_END,
    use_cache: bool = True,
) -> pd.DataFrame:
    """
    Загружает все макро-данные, объединяет в ежедневный датафрейм
    и сохраняет в RAW_DIR / cbr_macro_data.parquet.
    """
    cache_path = RAW_DIR / "cbr_macro_data.parquet"
    if use_cache and cache_path.exists():
        cached = pd.read_parquet(cache_path)
        if cached.index.min() <= pd.Timestamp(start) and cached.index.max() >= pd.Timestamp(end):
            logger.info("Используем кэш %s", cache_path)
            return cached.loc[start:end]

    logger.info("Загрузка макро-данных с %s по %s ...", start, end)

    usd_rub = fetch_usd_rub(start, end)
    logger.info("USD/RUB: %s записей (%s .. %s)", len(usd_rub), usd_rub.index.min().date(), usd_rub.index.max().date())

    key_rate = fetch_key_rate(start, end)
    logger.info("Key Rate: %s записей (%s .. %s)", len(key_rate), key_rate.index.min().date(), key_rate.index.max().date())

    inflation = fetch_inflation(start, end)
    logger.info("Inflation: %s записей (%s .. %s)", len(inflation), inflation.index.min().date(), inflation.index.max().date())

    m2 = fetch_m2(start, end)
    logger.info("M2: %s записей (%s .. %s)", len(m2), m2.index.min().date(), m2.index.max().date())

    # Ежедневная сетка
    dates = pd.date_range(start, end, freq="D")
    df = pd.DataFrame(index=dates)
    df["usd_rub"] = usd_rub.reindex(df.index).ffill().bfill()
    df["macro_key_rate"] = key_rate.reindex(df.index).ffill().bfill()
    df["macro_inflation"] = inflation.reindex(df.index).ffill().bfill()
    df["macro_money_supply"] = m2.reindex(df.index).ffill().bfill()

    # Сохраняем
    df.to_parquet(cache_path)
    logger.info("Макро-данные сохранены: %s (shape=%s)", cache_path, df.shape)
    return df

macro_df = prepare_macro_data("2010-01-01", "2024-12-31", use_cache=True)
macro_df = macro_df.reset_index().rename(columns={"index": "date"})
macro_df["date"] = pd.to_datetime(macro_df["date"])
macro_df

2026-05-22 10:23:15,534 [INFO] Используем кэш d:\Storage\Projects\dpo\dpo\project\bigdata\raw\cbr_macro_data.parquet


,date,usd_rub,macro_key_rate,macro_inflation,macro_money_supply
0,2010-01-01,30.1851,5.5,7.07,15267.6
1,2010-01-02,30.1851,5.5,7.07,15267.6
2,2010-01-03,30.1851,5.5,7.07,15267.6
3,2010-01-04,30.1851,5.5,7.07,15267.6
4,2010-01-05,30.1851,5.5,7.07,15267.6
...,...,...,...,...,...
5474,2024-12-27,99.2295,21.0,9.52,111025.2
5475,2024-12-28,100.5281,21.0,9.52,111025.2
5476,2024-12-29,101.6797,21.0,9.52,111025.2
5477,2024-12-30,101.6797,21.0,9.52,111025.2


In [ ]:
# Новости и сентимент
raw_news_path = RAW_DIR / "russian_news.csv"
if raw_news_path.exists():
    raw_news_path.unlink()

ds_train = load_dataset("Kasymkhan/RussianFinancialNews", split="train")
ds_test = load_dataset("Kasymkhan/RussianFinancialNews", split="test")
df = pd.concat([pd.DataFrame(ds_train), pd.DataFrame(ds_test)], ignore_index=True)
df.to_csv(raw_news_path, index=False)
df['date'] = pd.to_datetime(df['date'], format='mixed', errors='coerce')
df

2026-05-22 10:23:16,075 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/Kasymkhan/RussianFinancialNews/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-05-22 10:23:16,106 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Kasymkhan/RussianFinancialNews/f83da5c05c82c9086806cf90ded4155052f9605c/README.md "HTTP/1.1 200 OK"
2026-05-22 10:23:16,259 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/Kasymkhan/RussianFinancialNews/resolve/f83da5c05c82c9086806cf90ded4155052f9605c/RussianFinancialNews.py "HTTP/1.1 404 Not Found"
2026-05-22 10:23:16,801 [INFO] HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/Kasymkhan/RussianFinancialNews/Kasymkhan/RussianFinancialNews.py "HTTP/1.1 404 Not Found"
2026-05-22 10:23:16,947 [INFO] HTTP Request: GET https://huggingface.co/api/datasets/Kasymkhan/RussianFinancialNews/revision/f83da5c05c82c9086806cf90ded4155052f9605c "HTTP/1.1 200 OK"
2026-05-22 10:23:17,085 [INF

,title,body,date,time,tags,source,__index_level_0__
0,no title,Российские отели «всё включено» обяжут предлаг...,2023-03-10,not stated,no tags,smart_lab,68555
1,no title,""" Самолет (SMLT) в 4кв2023 собирается побить р...",2023-10-26,08:31:24,"['аналитика', 'факты']",rdv,2032
2,no title,"""Высокие цены на нефть, похоже, надолго. Мир в...",2022-08-29,11:56:47,"['ROSN', 'LKOH']",rdv,4891
3,no title,ЦБ определил порядок продажи заблокированных и...,2023-12-11,not stated,no tags,smart_lab,78666
4,Подборка замещающих облигаций в евро: доходнос...,Чем привлекателен выпуск Газпром Капитала,2024-01-12,13:51,['Газпром капитал ЗО28-1-Е RU000A105BY1'],bcs,39150
...,...,...,...,...,...,...,...
91950,no title,"Порядок трансформации ИИС утвержден, приказ вс...",2024-11-30,not stated,no tags,smart_lab,91816
91951,"Аэрофлот. Следующая цель взята, куда дальше",На предыдущей торговой сессии акции Аэрофлота ...,2022-10-28,not stated,Аэрофлот,bcs_tech,51579
91952,no title,"Доллар выглядит перегретым, поэтому его рост к...",2022-10-07,not stated,no tags,smart_lab,63590
91953,Завершается первый этап сделки по продаже акти...,"Кроме того, сегодня последний день для попадан...",2024-04-30,not stated,"['ЛСР ао', 'ЯНДЕКС']",finam,14518


Почему выбрана модель mxlcw/rubert-tiny2-russian-financial-sentiment
Модель дообучена на русскоязычных экономических новостях и Telegram-каналах, поэтому лучше улавливает рыночные настроения, чем общий BERT. Она лёгкая (29 МБ), работает быстро даже без GPU и выдаёт три класса: positive, negative, neutral, что идеально для расчёта дневного сентимента.

In [11]:
device = 0 if torch.cuda.is_available() else -1
pipe = pipeline(
            "text-classification",
            model="mxlcw/rubert-tiny2-russian-financial-sentiment",
            truncation=True,
            max_length=512,
            device=device,
            return_all_scores=False
        )

texts = df['body'].tolist()
scores = []
batch_size = 32
for i in tqdm(range(0, len(texts), batch_size), desc="Обработка новостей"):
    batch = texts[i:i + batch_size]
    for res in pipe(batch):
        label = res['label'].lower()
        score = res['score']
        if 'positive' in label:
            scores.append(score)
        elif 'negative' in label:
            scores.append(-score)
        else:
            scores.append(0.0)

df['sentiment_score'] = scores

2026-05-22 10:23:26,119 [INFO] HTTP Request: HEAD https://huggingface.co/mxlcw/rubert-tiny2-russian-financial-sentiment/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 10:23:26,161 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mxlcw/rubert-tiny2-russian-financial-sentiment/a02913e44597582218db7821d52dc15c331bf427/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 57/57 [00:00<?, ?it/s]
2026-05-22 10:23:26,376 [INFO] HTTP Request: GET https://huggingface.co/api/models/mxlcw/rubert-tiny2-russian-financial-sentiment/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-22 10:23:26,529 [INFO] HTTP Request: GET https://huggingface.co/api/models/mxlcw/rubert-tiny2-russian-financial-sentiment/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
Обработка новостей: 100%|██████████| 2874/2874 [06:58<00:00,  6.87it/s]


In [12]:
daily = df.groupby(df['date'].dt.date).agg(
    news_sentiment=('sentiment_score', 'mean'),
    news_count=('sentiment_score', 'count'),
    news_sentiment_std=('sentiment_score', 'std')
).reset_index()
daily['news_sentiment_std'] = daily['news_sentiment_std'].fillna(0)
daily["date"] = pd.to_datetime(daily["date"])
news_df = daily[['date', 'news_sentiment', 'news_count', 'news_sentiment_std']]
news_df

,date,news_sentiment,news_count,news_sentiment_std
0,2009-06-19,-0.659678,1,0.000000
1,2011-01-21,0.913883,1,0.000000
2,2011-04-05,-0.900699,1,0.000000
3,2011-04-06,0.955771,1,0.000000
4,2011-04-07,-0.773694,1,0.000000
...,...,...,...,...
2027,2024-12-12,0.214017,127,0.548575
2028,2024-12-13,0.128261,124,0.541843
2029,2024-12-14,0.131456,24,0.401759
2030,2024-12-15,0.083142,9,0.533572


In [ ]:
# Основной пайплайн подготовки данных
# Загрузка OHLCV и расчёт признаков
tickers = sorted({f.stem.replace("_D1","") for f in RAW_DIR.glob("*_D1.csv")})
all_data = []
for t in tickers:
    f = RAW_DIR / f"{t}_D1.csv"
    df = pd.read_csv(f)
    df.columns = [c.strip().lower() for c in df.columns]
    rename = {'<date>':'date','<datetime>':'date','<open>':'open','<high>':'high',
              '<low>':'low','<close>':'close','<vol>':'volume'}
    df.rename(columns=rename, inplace=True)
    date_col = 'date' if 'date' in df.columns else 'datetime'
    df['date'] = pd.to_datetime(df[date_col], errors='coerce')
    df.dropna(subset=['date'], inplace=True)
    df['ticker'] = t
    all_data.append(df[['date','open','high','low','close','volume','ticker']])
df = pd.concat(all_data, ignore_index=True)
df.sort_values(['ticker','date'], inplace=True)

In [14]:
# Удаляем строки с отрицательными или нулевыми ценами/объёмами
df = df[(df['close'] > 0) & (df['open'] > 0) & (df['high'] > 0) & (df['low'] > 0)]

In [15]:
# Технические индикаторы
# SMA
for p in SMA_PERIODS:
    df[f"sma_{p}"] = df.groupby("ticker")["close"].transform(lambda x: x.rolling(p).mean())
# EMA
for p in [MACD_FAST, MACD_SLOW]:
    df[f"ema_{p}"] = df.groupby("ticker")["close"].transform(lambda x: x.ewm(span=p, adjust=False).mean())
# RSI
def rsi_sma(series, period=RSI_PERIOD):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi
df["rsi"] = df.groupby("ticker")["close"].transform(lambda x: rsi_sma(x, RSI_PERIOD))
# MACD
ema_fast = df.groupby("ticker")["close"].transform(lambda x: x.ewm(span=MACD_FAST, adjust=False).mean())
ema_slow = df.groupby("ticker")["close"].transform(lambda x: x.ewm(span=MACD_SLOW, adjust=False).mean())
df["macd"] = ema_fast - ema_slow
df["macd_signal"] = df.groupby("ticker")["macd"].transform(lambda x: x.ewm(span=MACD_SIGNAL, adjust=False).mean())
df["macd_hist"] = df["macd"] - df["macd_signal"]
# Bollinger Bands
df["bb_mid"] = df.groupby("ticker")["close"].transform(lambda x: x.rolling(BB_PERIOD).mean())
bb_std = df.groupby("ticker")["close"].transform(lambda x: x.rolling(BB_PERIOD).std())
df["bb_upper"] = df["bb_mid"] + 2 * bb_std
df["bb_lower"] = df["bb_mid"] - 2 * bb_std
# ATR
df["close_shift"] = df.groupby("ticker")["close"].shift(1)
df["tr1"] = df["high"] - df["low"]
df["tr2"] = (df["high"] - df["close_shift"]).abs()
df["tr3"] = (df["low"] - df["close_shift"]).abs()
df["tr"] = df[["tr1", "tr2", "tr3"]].max(axis=1)
df["atr"] = df.groupby("ticker")["tr"].transform(lambda x: x.rolling(ATR_PERIOD).mean())
df.drop(["close_shift", "tr1", "tr2", "tr3", "tr"], axis=1, inplace=True)
# Лог-доходность и волатильность
df["log_ret"] = np.log(df["close"] / df.groupby("ticker")["close"].shift(1))
df["volatility_20d"] = df.groupby("ticker")["log_ret"].transform(lambda x: x.rolling(VOL_PERIOD_SHORT).std())
df["parkinson_vol_20d"] = np.sqrt(
    (1 / (4 * np.log(2)))
    * df.groupby("ticker")
    .apply(lambda g: (np.log(g["high"] / g["low"]) ** 2).rolling(VOL_PERIOD_SHORT).mean())
    .reset_index(level=0, drop=True)
)
# Маркер экстремальных дней (кризисы, корпоративные действия)
df['daily_change'] = (df['close'] / df.groupby('ticker')['close'].shift(1) - 1).abs()
df['is_extreme_move'] = (df['daily_change'] > 0.3).astype(int)  # >30% за день
# Не удаляем, а маркируем — важная информация для модели

# Volume
df["vol_ma20"] = df.groupby("ticker")["volume"].transform(lambda x: x.rolling(VOL_PERIOD_SHORT).mean())
df["vol_ma50"] = df.groupby("ticker")["volume"].transform(lambda x: x.rolling(VOL_PERIOD_LONG).mean())
df["vol_ratio_20"] = df["volume"] / df["vol_ma20"].replace(0, np.nan)
df["vol_ratio_50"] = df["volume"] / df["vol_ma50"].replace(0, np.nan)
# Отклонение цены от SMA
for p in SMA_PERIODS:
    df[f"price_sma{p}_dev"] = (df["close"] - df[f"sma_{p}"]) / df[f"sma_{p}"].replace(0, np.nan)
# Stochastic Oscillator
low_roll = df.groupby("ticker")["low"].transform(lambda x: x.rolling(STOCH_PERIOD).min())
high_roll = df.groupby("ticker")["high"].transform(lambda x: x.rolling(STOCH_PERIOD).max())
range_hl = (high_roll - low_roll).replace(0, np.nan)
df["stoch_k"] = 100 * (df["close"] - low_roll) / range_hl
df["stoch_d"] = df.groupby("ticker")["stoch_k"].transform(lambda x: x.rolling(STOCH_SMOOTH).mean())
# Williams %R
df["williams_r"] = -100 * (high_roll - df["close"]) / range_hl
# CCI
df["typical_price"] = (df["high"] + df["low"] + df["close"]) / 3
tp_sma = df.groupby("ticker")["typical_price"].transform(lambda x: x.rolling(CCI_PERIOD).mean())
tp_md = df.groupby("ticker")["typical_price"].transform(
    lambda x: x.rolling(CCI_PERIOD).apply(lambda y: np.mean(np.abs(y - np.mean(y))), raw=True)
)
df["cci"] = (df["typical_price"] - tp_sma) / (0.015 * tp_md.replace(0, np.nan))
df.drop(["typical_price"], axis=1, inplace=True)
# OBV
df["close_diff"] = df.groupby("ticker")["close"].diff()
df["volume_signed"] = np.where(
    df["close_diff"] > 0,
    df["volume"],
    np.where(df["close_diff"] < 0, -df["volume"], 0),
)
df["obv"] = df.groupby("ticker")["volume_signed"].cumsum()
df.drop(["close_diff", "volume_signed"], axis=1, inplace=True)
# Momentum / ROC
df["momentum_20"] = df.groupby("ticker")["close"].transform(lambda x: x / x.shift(MOMENTUM_PERIOD) - 1)
df["roc_20"] = df.groupby("ticker")["close"].transform(lambda x: (x - x.shift(MOMENTUM_PERIOD)) / x.shift(MOMENTUM_PERIOD))
# Rolling max / drawdown
df["rolling_max_20"] = df.groupby("ticker")["close"].transform(lambda x: x.rolling(VOL_PERIOD_SHORT).max())
df["drawdown_20d"] = (df["close"] - df["rolling_max_20"]) / df["rolling_max_20"].replace(0, np.nan)
# Внутридневные паттерны
df["intraday_range"] = (df["high"] - df["low"]) / df["close"].replace(0, np.nan)
df["upper_shadow"] = (df["high"] - df[["open", "close"]].max(axis=1)) / df["close"].replace(0, np.nan)
df["lower_shadow"] = (df[["open", "close"]].min(axis=1) - df["low"]) / df["close"].replace(0, np.nan)
df["body_size"] = (df["close"] - df["open"]).abs() / df["close"].replace(0, np.nan)
# Лаги цены и объёма
for lag in [1, 5, 20]:
    df[f"close_lag_{lag}d"] = df.groupby("ticker")["close"].shift(lag)
    df[f"log_ret_lag_{lag}d"] = df.groupby("ticker")["log_ret"].shift(lag)
df["volume_lag_1d"] = df.groupby("ticker")["volume"].shift(1)
df["volume_change_1d"] = df.groupby("ticker")["volume"].pct_change(1)
# Сигналы пересечения
df["sma_20_50_cross"] = (df["sma_20"] > df["sma_50"]).astype(int)
df["sma_50_200_cross"] = (df["sma_50"] > df["sma_200"]).astype(int)
df["price_above_sma20"] = (df["close"] > df["sma_20"]).astype(int)
df["macd_bullish"] = (df["macd_hist"] > 0).astype(int)
# Рыночные (маркет-нейтральные) фичи
market_ret = df.groupby("date")["log_ret"].mean().reset_index()
market_ret.columns = ["date", "market_log_ret_1d"]
df = df.merge(market_ret, on="date", how="left")
df["relative_log_ret_1d"] = df["log_ret"] - df["market_log_ret_1d"]
df["relative_return_20d"] = df.groupby("ticker")["relative_log_ret_1d"].transform(
    lambda x: x.rolling(VOL_PERIOD_SHORT).mean()
)

In [16]:
for h in [1, 5, 20, 200]:
    df[f"target_return_{h}d"] = df.groupby("ticker")["close"].transform(lambda x: np.log(x.shift(-h) / x))
    df[f"target_binary_{h}d"] = (df[f"target_return_{h}d"] > 0).astype(int)

In [17]:
df

,date,open,high,low,close,volume,ticker,sma_20,sma_50,sma_200,...,relative_log_ret_1d,relative_return_20d,target_return_1d,target_binary_1d,target_return_5d,target_binary_5d,target_return_20d,target_binary_20d,target_return_200d,target_binary_200d
0,2009-12-11,9.87,13.39,9.71,12.13,6003,ABIO,NaN,NaN,NaN,...,NaN,NaN,0.341055,1,0.588061,1,0.192166,1,-0.141453,0
1,2009-12-14,12.90,17.10,12.62,17.06,7966,ABIO,NaN,NaN,NaN,...,0.333137,NaN,0.181735,1,0.274999,1,-0.225197,0,-0.478717,0
2,2009-12-15,18.87,20.48,17.56,20.46,1193,ABIO,NaN,NaN,NaN,...,0.176368,NaN,-0.004900,0,0.052368,1,-0.242140,0,-0.667097,0
3,2009-12-16,24.60,24.60,17.68,20.36,16565,ABIO,NaN,NaN,NaN,...,-0.023689,NaN,-0.021848,0,-0.025872,0,-0.258638,0,-0.674655,0
4,2009-12-17,19.50,21.42,16.92,19.92,8656,ABIO,NaN,NaN,NaN,...,-0.018409,NaN,0.092019,1,-0.071254,0,-0.293051,0,-0.666400,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
766534,2024-08-21,8.46,8.55,8.40,8.41,110,ZVEZ,8.5925,9.1384,11.52330,...,-0.001131,-0.000178,-0.030177,0,NaN,0,NaN,0,NaN,0
766535,2024-08-22,8.37,8.48,8.10,8.16,210,ZVEZ,8.5620,9.0808,11.48760,...,0.007517,0.000656,-0.019803,0,NaN,0,NaN,0,NaN,0
766536,2024-08-23,8.07,8.15,7.94,8.00,402,ZVEZ,8.5145,9.0212,11.45295,...,0.007792,-0.000693,0.021027,1,NaN,0,NaN,0,NaN,0
766537,2024-08-26,8.07,8.34,8.07,8.17,84,ZVEZ,8.5030,8.9670,11.42225,...,-0.006100,0.000716,0.000000,0,NaN,0,NaN,0,NaN,0


In [18]:
# Добавление макро и новостей
df = df.merge(macro_df, on="date", how="left")
macro_cols = ["macro_key_rate", "usd_rub", "macro_inflation", "macro_money_supply"]
for col in macro_cols:
    if col in df.columns:
        df[col] = df.groupby("ticker")[col].transform(lambda x: x.ffill().bfill())

# Макро-спреды (считаем на уникальных датах, чтобы не было leakage между тикерами)
macro_uniq = df[["date"] + macro_cols].drop_duplicates("date").sort_values("date")
macro_uniq["m2_growth_3m"] = macro_uniq["macro_money_supply"].pct_change(63)
macro_uniq["real_rate"] = macro_uniq["macro_key_rate"] - macro_uniq["macro_inflation"]
macro_uniq["usd_rub_vol_20d"] = macro_uniq["usd_rub"].rolling(VOL_PERIOD_SHORT).std()
macro_uniq["usd_rub_ma20"] = macro_uniq["usd_rub"].rolling(VOL_PERIOD_SHORT).mean()
macro_uniq["usd_rub_dev_from_ma20"] = (macro_uniq["usd_rub"] - macro_uniq["usd_rub_ma20"]) / macro_uniq["usd_rub_ma20"].replace(0, np.nan)

macro_extra_cols = ["m2_growth_3m", "real_rate", "usd_rub_vol_20d", "usd_rub_dev_from_ma20"]
df = df.merge(macro_uniq[["date"] + macro_extra_cols], on="date", how="left")

# Лаги и дельты макро
macro_all_cols = macro_cols + macro_extra_cols
for col in macro_all_cols:
    if col in df.columns:
        df[f"{col}_lag1m"] = df.groupby("ticker")[col].shift(21)
        df[f"{col}_lag3m"] = df.groupby("ticker")[col].shift(63)
        df[f"{col}_delta1m"] = df.groupby("ticker")[col].diff(21)
        df[f"{col}_delta3m"] = df.groupby("ticker")[col].diff(63)

for c in df.columns:
    if "lag" in c or "delta" in c:
        df[c] = df.groupby("ticker")[c].transform(lambda x: x.ffill().bfill())

df = df.merge(news_df, on="date", how="left")
news_cols = ["news_sentiment", "news_count", "news_sentiment_std"]
for c in news_cols:
    if c in df.columns:
        df[c] = df.groupby("ticker")[c].transform(lambda x: x.ffill().bfill())

In [19]:
# Временные фичи
df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["day_of_month"] = df["date"].dt.day
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["days_to_month_end"] = df["date"].dt.days_in_month - df["date"].dt.day
df["days_from_month_start"] = df["date"].dt.day
df["day_of_week_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["day_of_week_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["is_quarter_end"] = (df["month"] % 3 == 0).astype(int)
df["is_month_end"] = df["date"].dt.is_month_end.astype(int)

In [20]:
df.describe()

,date,open,high,low,close,volume,sma_20,sma_50,sma_200,ema_12,...,day_of_month,week_of_year,days_to_month_end,days_from_month_start,day_of_week_sin,day_of_week_cos,month_sin,month_cos,is_quarter_end,is_month_end
count,766539,766539.000000,766539.000000,766539.000000,766539.000000,7.665390e+05,761823.000000,754442.000000,717577.000000,766539.000000,...,766539.000000,766539.000000,766539.000000,766539.000000,766539.000000,766539.000000,7.665390e+05,7.665390e+05,766539.000000,766539.000000
mean,2016-09-12 01:27:53.795071,1112.269361,1133.706187,1092.699810,1112.739594,5.514331e+06,1111.363009,1108.664957,1086.259566,1110.921373,...,16.017947,26.880855,14.435710,16.017947,0.350115,-0.087655,-1.446087e-02,-1.604038e-02,0.331733,0.031530
min,1999-06-01 00:00:00,0.000760,0.000780,0.000490,0.000740,-2.146965e+09,0.000804,0.000814,0.000905,0.000800,...,1.000000,1.000000,0.000000,1.000000,-0.974928,-0.900969,-1.000000e+00,-1.000000e+00,0.000000,0.000000
25%,2012-08-29 00:00:00,3.070000,3.150000,3.000000,3.071000,1.000000e+02,3.077500,3.081000,3.027300,3.079411,...,9.000000,14.000000,7.000000,9.000000,0.000000,-0.900969,-8.660254e-01,-8.660254e-01,0.000000,0.000000
50%,2017-07-20 00:00:00,54.300000,55.300000,53.020000,54.230000,1.525000e+03,54.350000,54.445700,54.411500,54.376450,...,16.000000,27.000000,14.000000,16.000000,0.433884,-0.222521,-2.449294e-16,-1.836970e-16,0.000000,0.000000
75%,2021-05-13 00:00:00,418.000000,426.000000,409.000000,418.000000,5.055400e+04,417.950000,416.234500,413.800000,418.427137,...,24.000000,40.000000,22.000000,24.000000,0.781831,0.623490,5.000000e-01,5.000000e-01,1.000000,0.000000
max,2024-08-27 00:00:00,107800.000000,107800.000000,107800.000000,107800.000000,2.147474e+09,94480.000000,89540.000000,79466.000000,95093.523638,...,31.000000,53.000000,30.000000,31.000000,0.974928,1.000000,1.000000e+00,1.000000e+00,1.000000,1.000000
std,NaN,4631.871758,4722.039219,4557.701005,4638.900418,1.167731e+08,4631.789908,4622.314118,4544.172274,4628.096195,...,8.720689,14.863926,8.699888,8.720689,0.520001,0.774168,7.076328e-01,7.062512e-01,0.470836,0.174745


In [28]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.sort_values(['date','ticker']).reset_index(drop=True)
df

,date,open,high,low,close,volume,ticker,sma_20,sma_50,sma_200,...,day_of_month,week_of_year,days_to_month_end,days_from_month_start,day_of_week_sin,day_of_week_cos,month_sin,month_cos,is_quarter_end,is_month_end
0,1999-06-01,228.500,228.500,223.100,225.000,12471,LKOH,NaN,NaN,NaN,...,1,22,29,1,0.781831,0.62349,1.224647e-16,-1.0,1,0
1,1999-06-01,0.675,0.690,0.659,0.660,2007,MSNG,NaN,NaN,NaN,...,1,22,29,1,0.781831,0.62349,1.224647e-16,-1.0,1,0
2,1999-06-01,28.220,28.220,28.000,28.000,2,RTKM,NaN,NaN,NaN,...,1,22,29,1,0.781831,0.62349,1.224647e-16,-1.0,1,0
3,1999-06-01,0.540,0.540,0.530,0.530,778,SBER,NaN,NaN,NaN,...,1,22,29,1,0.781831,0.62349,1.224647e-16,-1.0,1,0
4,1999-06-01,0.290,0.290,0.280,0.280,40200,SBERP,NaN,NaN,NaN,...,1,22,29,1,0.781831,0.62349,1.224647e-16,-1.0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
766534,2024-08-27,0.442,0.448,0.435,0.438,213,YKENP,0.4588,0.46594,0.580385,...,27,35,4,27,0.781831,0.62349,-8.660254e-01,-0.5,0,0
766535,2024-08-27,882.000,882.000,858.000,862.000,13,YRSB,888.2000,888.88000,1097.380000,...,27,35,4,27,0.781831,0.62349,-8.660254e-01,-0.5,0,0
766536,2024-08-27,219.000,220.500,217.000,220.500,60,YRSBP,223.0750,226.34000,266.877500,...,27,35,4,27,0.781831,0.62349,-8.660254e-01,-0.5,0,0
766537,2024-08-27,3150.000,3230.000,3115.000,3115.000,198,ZILL,3225.5000,3284.20000,3327.375000,...,27,35,4,27,0.781831,0.62349,-8.660254e-01,-0.5,0,0


In [27]:
df.to_parquet(PROCESSED_DIR / "combined_features.parquet", index=False)

# Сохранение списка признаков
exclude = {'open','high','low','close','volume','ticker','date'}
exclude |= {c for c in df.columns if c.startswith('target_')}
features = [c for c in df.columns if c not in exclude]
with open(PROCESSED_DIR / "feature_columns.txt", 'w') as fout:
    fout.write('\n'.join(features))